In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Load environment variables from .env file
openai_client = OpenAI()  # Initialize the OpenAI client

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [5]:
import json


LLM Zoomcamp 1.14 — The Agentic Loop

In [1]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [7]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [9]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course enrollment late join discovered the course"}
function_call: search {"query":"enroll after course started can I still join"}


In [10]:
it = 1
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course late enroll discover course can I join it registration enrollment policy"}
function_call: search {"query":"course FAQ discovered the course can I join late enrollment enrollment deadline"}
function_call: search {"query":"new student join course after start registration policy FAQ"}
iteration #2...
ASSISTANT:
Yes, you can still join.

If you want a certificate, make sure you submit your project while the course is still accepting submissions. You can also start learning and follow the materials even if you discovered the course late.

If you'd like, I can also help you with how to get started or explain the certificate requirements. Are there other areas you want to explore?


In [11]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [12]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install local model server FAQ"}
function_call: search {"query":"Olama locally run install FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from https://ollama.com/download
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should see a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If you get a connection erro

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from https://ollama.com/download\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should see a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error, restarting the server can help:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you w

In [13]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late registration discovered course can I still join"}
function_call: search {"query":"course FAQ enroll late join after course started"}
function_call: search {"query":"can I still join the course after it started FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join.

If you want a certificate, make sure to submit your project while submissions are still being accepted. You can also start learning and working through the materials right away.

If you want, I can also help with how to catch up quickly or explain the certificate requirements.


'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted. You can also start learning and working through the materials right away.\n\nIf you want, I can also help with how to catch up quickly or explain the certificate requirements.'

In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course discovered course can I join enrollment late registration FAQ"}
iteration #2...
function_call: search {"query":"accepting submissions project certificate while we're still accepting submissions FAQ late join certificate project"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions. You can start learning and working through the materials even if you discovered it late.

If you want, I can also help with how to get started or explain the certificate requirements.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions. You can start learning and working through the materials even if you discovered it late.\n\nIf you want, I can also help with how to get started or explain the certificate requirements.'

In [15]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess queen's gambit opening definition"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening move d4 d5 c4 FAQ"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic chess opening.

It starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn as a “gambit” to try to gain control of the center and create better position.

There are two main types:
- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain **why it’s called a gambit** or show the **basic ideas for both sides**.


'The **Queen’s Gambit** is a classic chess opening.\n\nIt starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn as a “gambit” to try to gain control of the center and create better position.\n\nThere are two main types:\n- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain **why it’s called a gambit** or show the **basic ideas for both sides**.'

In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit queen's gambit chess opening"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition faq"}
iteration #3...
ASSISTANT:
I couldn’t find any FAQ entry about the Queen’s Gambit as a chess opening, so this looks off-topic for the course FAQ.

If you meant a course-specific term or a logistics question, please rephrase it and I can look it up. Are there other areas you want to explore?


'I couldn’t find any FAQ entry about the Queen’s Gambit as a chess opening, so this looks off-topic for the course FAQ.\n\nIf you meant a course-specific term or a logistics question, please rephrase it and I can look it up. Are there other areas you want to explore?'

In [17]:
agent_loop(instructions, "I just find the program, can I wtill enroll?")

iteration #1...
function_call: search {"query":"just found the program can I still enroll enrollment late enrollment add course program"}
iteration #2...
function_call: search {"query":"enroll register join course accepted start learning without registering registration gauge interest confirmation email"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

You don’t need a confirmation email or approval to start. Registration is just to gauge interest, and you can begin learning and submitting homework while the form is open. If you want a certificate, make sure you submit your project while submissions are still being accepted.

Would you like help with anything else about the course?


'Yes — you can still join the course.\n\nYou don’t need a confirmation email or approval to start. Registration is just to gauge interest, and you can begin learning and submitting homework while the form is open. If you want a certificate, make sure you submit your project while submissions are still being accepted.\n\nWould you like help with anything else about the course?'